# Install packages

In [ ]:
!pip install swig mediapy
!pip install mujoco==3.1.4
!pip install torch torchrl
!pip install gymnasium[box2d,mujoco]==0.28.1

# Imports and Functions

In [ ]:
# Imports
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation, rc
from ipywidgets import interact, widgets
# Change to glfw when on Mac. This might fuck up the lighting.
%env MUJOCO_GL=egl
import mujoco
import mujoco.viewer
import mediapy as media
from time import time

import torch
from torch import nn
import torch.nn.functional as F
from tensordict import TensorDict
from torchrl.data import Composite, Bounded, UnboundedContinuous
from torchrl.envs.model_based import ModelBasedEnvBase
from torchrl.modules import SafeModule
from torchrl.modules import CEMPlanner
from torchrl.modules import WorldModelWrapper
from torchrl.data import TensorDictReplayBuffer

In [ ]:
def render_episode(policy, env, max_steps=1000):
	frames = []
	score = 0
	state = env.get_state()
	terminated = truncated = False
	while not terminated and not truncated and len(frames)<max_steps:
		frame = env.render()
		frames.append(frame)
		action = policy(state)
		state, reward, terminated, truncated, _ = env.step(action)
		score += reward
	env.close()
	print(f"Finished episode. Cummulative Return: {score}")
	media.show_video(frames, fps=50, loop=True)

In [ ]:
class MujocoCartPoleEnv(gym.Env):
    def __init__(self, m = 2, M = 5, l = 0.5, g = 9.81, k = 100, dt = 0.02, end_on_failure = True):
        self.model = self._get_mj_model(m,M,l,g,k,dt)
        self.data = mujoco.MjData(self.model)
        self.renderer = mujoco.Renderer(self.model)
        self.truncate = 1000
        self.action_space = gym.spaces.Box(low=-1, high=1, shape=(self.model.nu,))
        self.observation_space = gym.spaces.Box(low=-np.inf, high=np.inf, shape=(self.model.nq + self.model.nv,))
        self.end_on_failure = end_on_failure
        self.reset()

    def reset(self):
        mujoco.mj_resetData(self.model, self.data)
        self.data.qvel = 0.0*np.random.randn(2) # velocities
        self.data.qpos = 0.1*np.random.randn(2) # positions
        self.timestep = 0
        return self.get_state(), {}

    def set_state(self, state):
        self.data.qpos = state[:self.model.nq]
        self.data.qvel = state[self.model.nq:]

    def get_state(self):
        return np.concatenate([self.data.qpos, self.data.qvel])

    def step(self, action):
        self.data.ctrl = action
        mujoco.mj_step(self.model, self.data)
        next_state = self.get_state()
        reward = self._calculate_reward()
        terminal = self._is_done()
        truncated = self.timestep >= self.truncate
        info = {}  # Additional information (optional)
        self.timestep += 1
        return (next_state, reward, terminal, truncated, info)

    def render(self):
        self.renderer.update_scene(self.data)
        return self.renderer.render()

    def _calculate_reward(self):
        # Calculate reward based on the current state (optional)
        return int(self.data.qpos[1] > - np.pi/12 and self.data.qpos[1] < np.pi/6)

    def _is_done(self):
        # Check if the episode is done based on the current state (optional)
        if self.end_on_failure:
            return (self.data.qpos[1] < - np.pi/12 or self.data.qpos[1] > np.pi/6)
        return False

    # return a mujoco model with the given physical parameters
    @staticmethod
    def _get_mj_model(m,M,l,g,k,dt):
        xml = f"""
        <mujoco model='test_cartpole'>
            <compiler inertiafromgeom='true' coordinate='local'/>

            <size nkey="1"/>

            <option timestep='{dt}' integrator="RK4" gravity='0 0 {-g}'/>

            <default>
            <joint damping='0.0' solreflimit='.08 1'/>
            <geom contype='0' friction='0. 0. 0.'/>
            </default>

            <worldbody>
            <camera name='fixed' pos='0 -2.5 0' quat='0.707 0.707 0 0'/>
            <light name="top" castshadow="false"/>
            <geom name='floor' pos='0 0 -1' size='4 4 4' type='plane' />
            <geom name='rail1' type='capsule' pos='0 .07 0' quat='0.707 0 0.707 0'
                    size='0.02 2.2' />
            <geom name='rail2' type='capsule' pos='0 -.07 0' quat='0.707 0 0.707 0'
                    size='0.02 2.2' />
            <body name='cart' pos='0 0 0'>
                <camera name='cart' pos='0 -2.5 0' quat='0.707 0.707 0 0' />
                <joint name='slider' type='slide' limited='true' pos='0 0 0'
                        axis='1 0 0' range='-2 2' />
                <geom name='cart' type='box' pos='0 0 0'
                        mass='{M}' size='0.2 0.1 0.05' rgba='0.7 0.7 0 1' />
                <site name='cart sensor' type='box' pos='0 0 0'
                        size='0.2 0.1 0.05' rgba='0.7 0.7 0 0' />
                <body name='pole' pos='0 0 0'>
                <camera name='pole'  pos='0 -2.5 0' quat='0.707 0.707 0 0' />
                <joint name='hinge' type='hinge' pos='0 0 0' axis='0 1 0'/>
                <geom name='cpole' type='capsule' fromto='0 0 0 0 0 {l}'
                        mass='0' size='0.01 {l}' rgba='0 0.7 0.7 1' />
                <geom type='sphere' size='.05' name='tip' mass='{m}' pos='.001 0 {l}'/>
                </body>
            </body>
            </worldbody>

            <actuator>
            <motor name='slide' joint='slider' gear='{k}' ctrllimited='true' ctrlrange='-1 1' />
            </actuator>

        </mujoco>
        """
        return mujoco.MjModel.from_xml_string(xml)

# Model-Based Reinforcement Learning
Model-based reinforcement learning (MBRL) is a powerful approach that combines the principles of reinforcement learning with the use of predictive models. In MBRL, an agent learns a model of the environment dynamics and uses this model to plan and make decisions. By explicitly representing the dynamics of the environment, MBRL enables the agent to simulate different scenarios and evaluate the potential outcomes of different actions. This allows the agent to make more informed decisions and optimize its behavior in complex and uncertain environments. In contrast to model-free reinforcement learning, which learns directly from trial and error, MBRL leverages the learned model to plan ahead and make more efficient use of the available data. Also, the learned model is task-agnostic, so that an MBRL agent does not have to learn from scratch for new tasks. However, MBRL also comes with its own set of challenges. Building an accurate model of the environment can be difficult, and errors in the model can lead to suboptimal performance. Additionally, the planning process in MBRL can be computationally expensive, especially in large and complex environments. Despite these challenges, MBRL offers the potential for improved sample efficiency and better performance in certain domains.

In [ ]:
# Objective function
def quadratic_cost(trajectory, desired_state = torch.zeros(4), weight = torch.tensor([1.,2.,0.,0.])):
	with torch.no_grad():
		return torch.square(trajectory-desired_state)@weight

# Wrapper for the world model
class MyTorchRLModel(ModelBasedEnvBase):
    def __init__(self, world_model, device="cpu", batch_size=None):
        super().__init__(world_model, device=device, batch_size=batch_size)
        self.state_spec = Composite(
            state=UnboundedContinuous((4,))
        )
        self.observation_spec = Composite(
            state=UnboundedContinuous((4,))
        )
        self.action_spec = Bounded(-1,1,(1,))
        self.reward_spec = UnboundedContinuous((1,))

    def _reset(self, tensordict: TensorDict) -> TensorDict:
        tensordict = TensorDict(
            source={'state': torch.tensor([0.,0.,0.,0.]),
                    'action': torch.tensor([0.])},
            batch_size=self.batch_size,
            device=self.device,
        )
        return tensordict

## Learning the forward model
Learning a forward dynamics model using supervised learning involves training a model to predict the future state of a system given its current state and the applied control inputs. In this approach, a dataset is collected by executing the system under different control inputs and recording the corresponding state transitions. The dataset is then used to train a supervised learning model, such as a neural network, to learn the underlying dynamics of the system. The model is trained to minimize the prediction error between the predicted future state and the actual future state. Once trained, the forward dynamics model can be used to simulate the behavior of the system and make predictions about its future states under different control inputs. This approach is particularly useful in model-based reinforcement learning, where the learned forward dynamics model can be used for planning and decision-making purposes.

In [ ]:
## Solution
# Neural Network as forward dynamics and reward model
class ForwardDynamicsModel(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dim = 256):
        super(ForwardDynamicsModel, self).__init__()
        self.fc1 = nn.Linear(state_dim+action_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, state_dim)

    def forward(self, state, action):
        x = torch.cat((state, action), dim=1)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x) + state
        return x

In [ ]:
## Solution
# we define an agent that can act and learn
class MBRLAgent():
    def __init__(self, state_size, action_size, seed = 42, device = 'cpu'):
        self.state_size = state_size
        self.action_size = action_size
        self.batch_size = 64
        self.device = device
        self.replay_buffer = TensorDictReplayBuffer(batch_size=self.batch_size)
        self.model = ForwardDynamicsModel(state_size, action_size)
        world_model = WorldModelWrapper(
            SafeModule(
                self.model,
                in_keys=["state", "action"],
                out_keys=["state"],
            ),
            SafeModule(
                lambda traj: - quadratic_cost(traj),
                in_keys=["state"],
                out_keys=["reward"],
            ),
        )
        plan_model = MyTorchRLModel(world_model, device=device)
        self.planner = CEMPlanner(plan_model, 20, 4, 1000, 3)
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=0.0001)
        self.init_period = 500
        self.update_every = 4
        self.update_steps = 16
        self.t_step = 0

    def step(self, state, action, reward, next_state, done):
        self.replay_buffer.add(
            TensorDict({
                'state': torch.from_numpy(state),
                'action': torch.from_numpy(action),
                'next_state': torch.from_numpy(next_state)
                }
            ),
        )
        # Learn every update_every time steps.
        if self.t_step % self.update_every == 0:
            if self.t_step > self.init_period:
                for _ in range(self.update_steps):
                    self.learn()
        self.t_step += 1

    def act(self, state, eps=0.):
        self.model.eval()
        if self.t_step < self.init_period:
            return np.random.uniform(-1, 1, (1,)).astype(np.float32)
        td = TensorDict({'action': torch.tensor([0.]),
					 'done': torch.tensor([False]),
					 'state': torch.tensor(state, dtype=torch.float32).to(self.device),
					 'terminated': torch.tensor([False])})
        return self.planner(td)['action'].detach().numpy()

    def learn(self):
        self.model.train()
        td = self.replay_buffer.sample(self.batch_size)
        next_state_predcited = self.model(td['state'], td['action'])
        loss = F.mse_loss(next_state_predcited, td['next_state'])
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

In [ ]:
# Training Loop
agent = MBRLAgent(4, 1)
env = MujocoCartPoleEnv(end_on_failure=True)
scores = []
for i in range(100):
    state,_ = env.reset()
    score = 0
    done = False
    while not done:
        action = agent.act(state)
        next_state, reward, terminal, truncated, _ = env.step(action)
        done = terminal or truncated
        agent.step(state.astype(np.float32), action.astype(np.float32), reward, next_state.astype(np.float32), done)
        state = next_state
        score += reward
    print(f"Episode {i}, Score: {score}")
    scores.append(score)
    if len(scores)>5 and np.mean(scores[-5:]) >= 700:
        break

In [ ]:
fig, ax = plt.subplots()
start_learning = np.where(np.cumsum(scores) > agent.init_period)[0][0]
ax.plot(scores, label='Episode Length')
ax.axvline(start_learning, color='r', linestyle='--', label='Start Learning')
ax.set_xlabel('Episode')
ax.set_ylabel('Episode Length')
fig.legend()

In [ ]:
env = MujocoCartPoleEnv(end_on_failure=True)
env.reset()
render_episode(agent.act, env) # Render an episode

### Tasks:
1. Implement the forward model as a neural network.
2. Implement the learn function of the MBRL agent. You can choose a loss function. An easy choice would be the MSE of the
3. Compare the performance of Model-Based RL and Model-Free RL agents (DQN, REINFORCE, AC,... are Model-Free) in terms of wall-clock time and data-efficiency.
4. What issues do you see when training a forward model with an MSE loss function for optimizing rewards?

# Policy Distillation
Policy distillation is a technique in reinforcement learning that aims to transfer knowledge from a high-performing "teacher" policy to a "student" policy. The teacher policy is typically a complex and well-trained policy that has achieved good performance in a given task. The goal of policy distillation is to distill the knowledge and expertise of the teacher policy into a simpler and more computationally efficient student policy. This is done by training the student policy to mimic the behavior of the teacher policy, either by directly imitating its actions or by learning from its policy logits. Policy distillation can be particularly useful in scenarios where the teacher policy is too computationally expensive to deploy in real-time or when the teacher policy's knowledge needs to be transferred to a resource-constrained environment. By distilling the teacher policy's knowledge into a student policy, we can achieve comparable performance with reduced computational requirements, making it a valuable technique in reinforcement learning.

In [ ]:
# Neural Network Policy
class DeterministicPolicy(nn.Module):
    def __init__(self, obs_space_dims: int, action_space_dims: int, hidden_space:int=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_space_dims, hidden_space),
            nn.ELU(),
            nn.Linear(hidden_space, action_space_dims),
        )

    def forward(self, x: torch.Tensor):
        return self.net(x)

In [ ]:
# Distill the agents policy into a neural network policy
def distill_policy(agent, policy, batches=30, updates_per_batch=40):

    optim = torch.optim.Adam(policy.parameters(), lr=0.05)
    policy.train()
    for i in range(batches):
        batch = agent.replay_buffer.sample(agent.batch_size)
        states = batch['state']
        mbrl_actions = torch.stack([torch.from_numpy(agent.act(state)) for state in states])
        for j in range(updates_per_batch):
            policy_actions = policy(states)
            loss = F.mse_loss(policy_actions, mbrl_actions)
            optim.zero_grad()
            loss.backward()
            optim.step()
        print(f"Batch {i} - Loss: {loss.item():.2f}")

In [ ]:
# create and distill nn policy
policy = DeterministicPolicy(4, 1)
distill_policy(agent,policy)

In [ ]:
# Run the distilled policy
env = MujocoCartPoleEnv(end_on_failure=True)
env.reset()
policy.eval()
render_episode(lambda s: policy(torch.from_numpy(s.astype(np.float32))).detach().cpu().numpy(), env) # Render an episode

### Tasks
1. Compare the inference time for the CEM planner and the NN policy.
2. We have seen that model-based RL algorithms are a lot more data efficient than model-free ones. However, model-free algorithms are computationally a lot more efficient. Can you think of a good combination of the two approaches? Code an agent that simultaneously learns a forward model and a policy (and maybe even a value function). Try to let it converge to a score of 1000 in no more than 100 episodes while having an inference time of no more than 50 ms. You can use the quadratic cost function.

In [ ]:
## Solution
# we define an agent that can act and learn
class PolicyPlanningAgent():
    def __init__(self, state_size, action_size, seed = 42, device = 'cpu'):
        self.state_size = state_size
        self.action_size = action_size
        self.batch_size = 64
        self.device = device
        self.replay_buffer = TensorDictReplayBuffer(batch_size=self.batch_size)
        self.model = ForwardDynamicsModel(state_size, action_size)
        self.policy = DeterministicPolicy(state_size, action_size)
        self.model_optimizer = torch.optim.Adam(self.model.parameters(), lr=0.0001)
        self.policy_optimizer = torch.optim.Adam(self.policy.parameters(), lr=0.05)
        self.update_model_every = 4
        self.update_model_steps = 16
        self.update_policy_every = 256
        self.update_policy_steps = 32
        self.init_period = 500
        self.t_step = 0

    def step(self, state, action, reward, next_state, done):
        self.replay_buffer.add(
            TensorDict({
                'state': torch.from_numpy(state),
                'action': torch.from_numpy(action),
                'next_state': torch.from_numpy(next_state)
                }
            ),
        )
        # Learn every update_every time steps.
        if self.t_step > self.init_period:
            self.learn()
        self.t_step += 1

    def _quadratic_cost(self, trajectory, desired_state = torch.zeros(4), weight = torch.tensor([1.,10.,0.,0.])):
        with torch.no_grad():
            return torch.sum(torch.square(trajectory-desired_state)@weight,dim=1)

    def plan(self, state, horizon = 20, num_candidates=5000, eps=0.2):
        states = torch.tile(state, (num_candidates,1))
        trajectory = torch.zeros((num_candidates, horizon, self.state_size))
        first_actions = torch.rand((num_candidates, self.action_size))*2-1
        actions = first_actions
        for t in range(horizon):
            states = self.model(states, actions)
            trajectory[:,t,:] = states
            noise = torch.randn((num_candidates, self.action_size))*eps
            actions = self.policy(states) + noise
        costs = self._quadratic_cost(trajectory)
        best_cand = costs.argmin()
        return first_actions[best_cand]

    def act(self, state, eps=0.):
        self.model.eval()
        self.policy.eval()
        action = self.plan(torch.from_numpy(state.astype(np.float32))).detach().numpy()
        # noise = np.random.randn()*eps
        return np.clip(action,-1,1)

    def learn(self):
        # forward model
        if self.t_step % self.update_model_every == 0:
            self.model.train()
            for _ in range(self.update_model_steps):
                batch = self.replay_buffer.sample(self.batch_size)
                next_state_predcited = self.model(batch['state'], batch['action'])
                loss = F.mse_loss(next_state_predcited, batch['next_state'])
                self.model_optimizer.zero_grad()
                loss.backward()
                self.model_optimizer.step()

        # policy
        if self.t_step % self.update_policy_every == 0:
            self.policy.train()
            batch = self.replay_buffer[-self.update_policy_every:]
            for _ in range(self.update_policy_steps):
                policy_actions = self.policy(batch['state'])
                loss = F.mse_loss(policy_actions, batch['action'])
                self.policy_optimizer.zero_grad()
                loss.backward()
                self.policy_optimizer.step()

In [ ]:
# Training Loop
agent = PolicyPlanningAgent(4, 1)
env = MujocoCartPoleEnv(end_on_failure=True)
scores = []
for i in range(100):
    inference_time = 0.
    state,_ = env.reset()
    score = 0
    done = False
    t = 0
    while not done:
        t1 = time()
        action = agent.act(state)
        inference_time += time()-t1
        next_state, reward, terminal, truncated, _ = env.step(action)
        done = terminal or truncated
        agent.step(state.astype(np.float32), action.astype(np.float32), reward, next_state.astype(np.float32), done)
        state = next_state
        score += reward
        t += 1
    print(f"Episode {i}, Score: {score}, avg. inference time: {1000.*inference_time/t:.2f}ms")
    scores.append(score)
    if len(scores)>5 and np.mean(scores[-5:]) >= 700:
        break

In [ ]:
# Plot results
fig, ax = plt.subplots()
start_learning = np.where(np.cumsum(scores) > agent.init_period)[0][0]
ax.plot(scores, label='Episode Length')
ax.axvline(start_learning, color='r', linestyle='--', label='Start Learning')
ax.set_xlabel('Episode')
ax.set_ylabel('Episode Length')
fig.legend()

In [ ]:
# Render episode using the planner
env = MujocoCartPoleEnv(end_on_failure=True)
env.reset()
render_episode(agent.act, env) # Render an episode

In [ ]:
# Render episode using the policy
env = MujocoCartPoleEnv(end_on_failure=True)
env.reset()
agent.policy.eval()
render_episode(lambda s: agent.policy(torch.from_numpy(s.astype(np.float32))).detach().cpu().numpy(), env) # Render an episode